# AyyPhoto — modèle 01 pour Kaggle

Variante pour un **notebook Kaggle** ou un éditeur connecté au serveur Jupyter
Kaggle. Aucun montage Google Drive n'est utilisé. Le dataset attaché est lu
directement sous `/kaggle/input` ; les résultats sont écrits sous
`/kaggle/working/ayyphoto/experiments`.

## Avant Run All

1. Dans **Add Input**, ajouter le dataset Kaggle
   **soumikrakshit/div2k-high-resolution-images**.
2. Activer un **GPU disponible** dans les options du notebook Kaggle.
3. Lancer **Run All**. Le dossier DIV2K est détecté automatiquement, y compris
   dans les arborescences imbriquées. S’il n’est pas trouvé, KaggleHub récupère
   directement le chemin du dataset Kaggle demandé, sans montage Drive. S'il y a plusieurs copies attachées,
   renseigner `DATA_ROOT` dans la section 2 pour choisir explicitement.

Les bibliothèques préinstallées de Kaggle sont utilisées par défaut. Le dataset
est lu depuis les entrées attachées ou depuis le chemin retourné par KaggleHub.
Aucune copie du dataset ni authentification Drive ne sont utilisées. L'installation pip est optionnelle et nécessite
l'accès Internet si tu l'actives.

**Ordre :** environnement → configuration → audit → aperçu → modèle →
entraînement → validation → test final optionnel → photo réelle → rapport.
Le modèle, les paramètres et les métriques sont ceux du notebook 01 : CNN
résiduel, 30 époques maximum, batch 8 et répartition 720 / 80 / 100. Le test
final reste désactivé pendant les essais.

## Conserver les résultats

Les checkpoints `last.ckpt`, `best_weights.pth`, les courbes et les rapports
seront sous `/kaggle/working/ayyphoto/experiments`. Avant de quitter la session,
enregistrer une version Kaggle avec ses sorties ou télécharger le dossier des
résultats. Ces fichiers ne sont pas envoyés automatiquement sur Google Drive.
Ne pas supposer que les fichiers d'une session interactive seront conservés
sans sauvegarde explicite.

Pour reprendre une expérience, renseigner `RESUME_RUN_DIR` avec son dossier
contenant `config.json`, `best_weights.pth` et `last.ckpt`. Les artefacts
d’une entrée en lecture seule sont copiés automatiquement sous le dossier
des expériences. Le dataset n’est jamais recopié. Pour seulement évaluer,
mettre `RUN_TRAINING = False`.

Un fichier `model_bundle.pt` et une archive `*_export.zip` sont générés
après chaque époque complète par défaut et à la fin. **Ce sont des fichiers
locaux de la session, pas une sauvegarde automatique hors de Kaggle.**
Il faut enregistrer une version avec ses sorties ou télécharger l’archive.
Pour tester un poids sauvegardé sans DIV2K et sans entraînement, utiliser
`02_ayyphoto_inference_kaggle.ipynb`.

La qualité du modèle sur le bruit réel des smartphones reste à évaluer.
Aucune cellule de cette variante n'a été exécutée par l'agent.


## 1. Bibliothèques et environnement Kaggle

`INSTALL_DEPENDENCIES = False` par défaut : conserver les bibliothèques de
l'environnement Kaggle. Si un import manque, activer Internet dans les options
Kaggle et cette installation, puis redémarrer le noyau si nécessaire.
L'installation conserve PyTorch s'il est déjà présent.


In [ ]:
import importlib.util
import subprocess
import sys

INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    packages = [
        "numpy>=1.26,<3", "Pillow>=10,<13", "matplotlib>=3.8,<4",
        "pandas>=2.1,<4", "opencv-python-headless>=4.8,<5",
        "scikit-image>=0.22,<1", "tqdm>=4.66,<5",
    ]
    if importlib.util.find_spec("torch") is None:
        packages.append("torch>=2.3,<3")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
else:
    print("Installation désactivée. Activer INSTALL_DEPENDENCIES si nécessaire.")


In [ ]:
from pathlib import Path

IN_KAGGLE = Path("/kaggle/input").is_dir() and Path("/kaggle/working").is_dir()
DATASET_PROVENANCE = None
print("Environnement :", "Kaggle" if IN_KAGGLE else "autre environnement — adapter les chemins")
print("Dataset : entrée Kaggle attachée ; aucun montage Drive requis.")


## 2. Configuration — à adapter avant le lancement

Cette variante utilise les **900 images DIV2K attachées au notebook Kaggle**.
Utiliser **Add Input** pour ajouter `soumikrakshit/div2k-high-resolution-images`.
Le dossier est détecté sous `/kaggle/input`. Organisation attendue dans le dataset :

```text
data/archive/
  DIV2K_train_HR/DIV2K_train_HR/0001.png  ... 0800.png
  DIV2K_valid_HR/DIV2K_valid_HR/0801.png  ... 0900.png
```

Les dossiers simples ou imbriqués sont reconnus récursivement.
Le dataset est lu sans être déplacé ou recopié. Répartition
reproductible avec la seed 42 : **720 train / 80 validation / 100 test final**.
Les 80 images de validation interne proviennent des 800 images d'entraînement
officielles. Les 100 images de validation publiques restent réservées au test
final de notre expérience : **ce n'est pas le test officiel DIV2K**.

DIV2K ne fournit pas ici de regroupement par scène : l'identifiant de l'image
sert de groupe, pas de preuve d'indépendance des scènes. Vérifier les contenus
similaires manuellement. Les conditions publiées par la
[source officielle](https://data.vision.ee.ethz.ch/cvl/DIV2K/) limitent la mise
à disposition à la recherche académique ; l'utilisation pour un produit
commercial nécessite de clarifier les droits avant réutilisation.

Pour un autre dataset, choisir `dataset_layout = "scene_folders"` et préparer :

```text
data/hq/
  train/scene_001/photo_01.png
  train/scene_002/photo_01.jpg
  val/scene_101/photo_01.png
  test/scene_201/photo_01.png
```

Chaque sous-dossier immédiat représente une scène/session. Toutes les photos
similaires d'une scène restent dans **un seul ensemble** ; les identifiants de
scène doivent être distincts entre `train`, `val` et `test`. Les doublons exacts
seront bloqués ; les quasi-doublons nécessitent une revue manuelle. Les photos
doivent avoir un petit côté d'au moins `eval_size` pixels (256 par défaut).

`DATA_ROOT = None` active la détection automatique du dataset attaché.
Pour choisir un chemin précis, remplacer `None` par `Path("/kaggle/input/...")`.
`OUTPUT_ROOT` vise `/kaggle/working/ayyphoto/experiments`. Aucun fichier
n’est écrit dans le dataset d’entrée et aucune copie du dataset n’est créée.
Une image par ensemble suffit pour vérifier le pipeline, mais pas pour juger la
généralisation. Le notebook n'inclut aucune photo d'exemple faisant office de dataset.

Pour commencer : laisser le test final désactivé, mettre `RUN_TRAINING = True`
quand tout est prêt, puis exécuter les cellules dans l'ordre. Si la mémoire GPU
est insuffisante, réduire `batch_size` à 4 ou 2. Pour reprendre une expérience,
renseigner `RESUME_RUN_DIR` avec le dossier contenant `last.ckpt` et augmenter
`epochs` si nécessaire. Réutiliser le même dataset, le même environnement et la
même configuration ; la reprise se fait à la dernière époque terminée.


In [ ]:
def find_div2k_input(input_root):
    input_root = Path(input_root)
    if not input_root.is_dir():
        raise FileNotFoundError(f"Dossier des entrées absent : {input_root}. Ouvrir ce notebook sur Kaggle.")
    candidates = set()
    for train_folder in input_root.rglob("DIV2K_train_HR"):
        if not train_folder.is_dir():
            continue
        parent = train_folder.parent
        if any(folder.is_dir() for folder in parent.rglob("DIV2K_valid_HR")):
            candidates.add(parent.resolve())
    if not candidates:
        raise FileNotFoundError(
            "DIV2K introuvable sous /kaggle/input. Dans Add Input, ajouter "
            "soumikrakshit/div2k-high-resolution-images, puis relancer cette cellule."
        )
    if len(candidates) != 1:
        raise ValueError(
            "Plusieurs datasets DIV2K détectés : " + ", ".join(str(path) for path in sorted(candidates))
            + ". Renseigner DATA_ROOT explicitement pour choisir."
        )
    return candidates.pop()

PROJECT_ROOT = Path("/kaggle/working/ayyphoto")
DATA_ROOT = None  # Auto-détection ; ou Path("/kaggle/input/<dossier-de-ton-dataset>").
OUTPUT_ROOT = PROJECT_ROOT / "experiments"
if DATA_ROOT is None:
    try:
        DATA_ROOT = find_div2k_input(Path("/kaggle/input"))
    except FileNotFoundError:
        if not IN_KAGGLE:
            raise
        import kagglehub
        print("Résolution du dataset Kaggle avec KaggleHub, sans Google Drive...")
        DATA_ROOT = Path(kagglehub.dataset_download(
            "soumikrakshit/div2k-high-resolution-images"
        )).resolve()
else:
    DATA_ROOT = Path(DATA_ROOT).expanduser().resolve()
REQUIRE_CUDA = True  # Activer un GPU dans les options Kaggle avant Run All.

RUN_TRAINING = True
RUN_FINAL_TEST = False
RESUME_RUN_DIR = ""  # Exemple : "/kaggle/working/ayyphoto/experiments/denoise_..."
REAL_PHOTO_PATH = ""  # Optionnel : une photo bruitée réelle JPEG/PNG.
RGB_CACHE_MIB = 64  # Cache RGB uint8 borné, par dataset et par processus.
EXPORT_EACH_EPOCH = True  # ZIP local des checkpoints ; ne sauvegarde pas la session Kaggle.


CFG = {
    "seed": 42,
    "model_version": "0.1.0-candidate",
    "dataset_version": "div2k_hr_split_v1",
    "dataset_layout": "div2k",  # Alternative : "scene_folders" pour data/hq.
    "div2k_val_fraction": 0.10,  # 80 des 800 images train officielles.
    "patch_size": 128,
    "eval_size": 256,
    "patches_per_image": 8,
    "batch_size": 8,
    "epochs": 30,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "channels": 48,
    "depth": 8,
    "train_sigma_min": 0.0,
    "train_sigma_max": 50.0,
    "eval_sigmas": [15.0, 25.0, 50.0],  # Échelle 0..255, fixée avant les essais.
    "patience": 8,
    "amp": True,  # Activé uniquement si CUDA est disponible.
    "num_workers": 0,  # Portable VS Code/Windows/Colab et reprise plus simple.
    "tile_size": 256,
    "tile_halo": 16,
}
assert CFG["epochs"] > 0 and CFG["batch_size"] > 0
assert CFG["dataset_layout"] in {"div2k", "scene_folders"}
assert 0 < CFG["div2k_val_fraction"] < 1
assert CFG["eval_size"] >= CFG["patch_size"] >= 8
assert CFG["depth"] >= 2 and CFG["channels"] > 0
assert CFG["tile_halo"] >= CFG["depth"], "Halo insuffisant pour les convolutions."
assert 0 <= CFG["train_sigma_min"] < CFG["train_sigma_max"]
assert all(0 < s <= CFG["train_sigma_max"] for s in CFG["eval_sigmas"])
assert len(CFG["eval_sigmas"]) > 0 and CFG["tile_size"] > 0
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"Données introuvables sur le noyau courant : {DATA_ROOT}. "
        "Ajouter DIV2K avec Add Input dans Kaggle, puis vérifier DATA_ROOT. "
        "Cette variante lit directement les entrées Kaggle sans téléchargement ni copie."
    )
DATASET_PROVENANCE = {
    "provider": "kaggle_dataset",
    "handle": "soumikrakshit/div2k-high-resolution-images",
    "path": str(DATA_ROOT),
    "access": "direct_read_no_dataset_download_or_copy",
}
print("Images :", DATA_ROOT)
print("Résultats :", OUTPUT_ROOT)
print("Entraînement activé :", RUN_TRAINING, "| Test final activé :", RUN_FINAL_TEST)


## 3. Imports, matériel et reproductibilité

CUDA est utilisé s'il est disponible ; le CPU fonctionne aussi, plus lentement.
Les générateurs aléatoires et l'ordre des données sont fixés. Cela n'assure pas
des résultats identiques entre versions de bibliothèques ou matériels différents.
Le manifeste et l'environnement sont enregistrés pour interpréter les résultats.


In [ ]:
import hashlib
import shutil
import zipfile
from collections import OrderedDict
import io
import json
import math
import platform
import random
import time
from datetime import datetime, timezone

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageCms, ImageOps
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from IPython.display import display

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 3):
    raise RuntimeError("PyTorch 2.3+ requis. Installer une version adaptée au CPU/GPU.")

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if REQUIRE_CUDA and DEVICE.type != "cuda":
    raise RuntimeError(
        "CUDA indisponible. Dans les options du notebook Kaggle, sélectionner un GPU, "
        "puis relancer les cellules après le redémarrage de la session."
    )
AMP_ENABLED = CFG["amp"] and DEVICE.type == "cuda"
ENVIRONMENT = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": str(torch.__version__),
    "cuda_runtime": torch.version.cuda,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "gpu_memory_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)
                      if DEVICE.type == "cuda" else None,
    "amp_enabled": AMP_ENABLED,
}
print(json.dumps(ENVIRONMENT, indent=2, ensure_ascii=False))


## 4. Audit des images et gel du manifeste

Lecture JPEG/PNG, application de l'orientation EXIF et conversion des profils ICC
vers sRGB lorsqu'ils existent. Les fichiers sources restent inchangés. Sans
profil ICC, les valeurs sont supposées être sRGB. Les images corrompues, trop
petites, avec transparence ou profils invalides provoquent une erreur explicite.

Le manifeste mémorise les empreintes des fichiers et des pixels RGB décodés.
Cela détecte les doublons exacts, pas les versions recadrées ou recompressées.
Pour DIV2K, l'inventaire exige les identifiants 0001–0800 et 0801–0900 ;
les identifiants d'image servent de groupes. La répartition s'effectue avant
tout recadrage et avant génération du bruit. Le manifeste enrichi est enregistré
au lancement d'une expérience, sans réorganiser `data/archive`.
Le test n'est ni prévisualisé ni évalué pendant le réglage du modèle ; ses fichiers
sont uniquement audités ici pour vérifier la séparation des données.


In [ ]:
def load_rgb(path):
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source)
        if image.mode in ("RGBA", "LA") or "transparency" in image.info:
            raise ValueError("Image avec transparence : fournir une photo RGB opaque.")
        profile = image.info.get("icc_profile")
        if profile:
            image = ImageCms.profileToProfile(
                image, ImageCms.ImageCmsProfile(io.BytesIO(profile)),
                ImageCms.createProfile("sRGB"), outputMode="RGB",
            )
        else:
            image = image.convert("RGB")
        return np.array(image, dtype=np.uint8, copy=True)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def dataset_candidates(root):
    candidates = {split: [] for split in ("train", "val", "test")}
    extensions = {".jpg", ".jpeg", ".png"}
    if CFG["dataset_layout"] == "div2k":
        paths = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
        train_paths, valid_paths, unknown = [], [], []
        for path in paths:
            parents = path.relative_to(root).parts[:-1]
            if "DIV2K_train_HR" in parents and "DIV2K_valid_HR" not in parents:
                train_paths.append(path)
            elif "DIV2K_valid_HR" in parents and "DIV2K_train_HR" not in parents:
                valid_paths.append(path)
            else:
                unknown.append(str(path))
        if unknown:
            raise ValueError(f"Images hors des dossiers DIV2K attendus : {unknown[:5]}")
        expected_train = {f"{index:04d}" for index in range(1, 801)}
        expected_valid = {f"{index:04d}" for index in range(801, 901)}
        if len(train_paths) != 800 or {p.stem for p in train_paths} != expected_train:
            raise ValueError("DIV2K_train_HR doit contenir les 800 images uniques 0001 à 0800.")
        if len(valid_paths) != 100 or {p.stem for p in valid_paths} != expected_valid:
            raise ValueError("DIV2K_valid_HR doit contenir les 100 images uniques 0801 à 0900.")
        validation_count = round(len(train_paths) * CFG["div2k_val_fraction"])
        if not 0 < validation_count < len(train_paths):
            raise ValueError("La fraction de validation doit conserver au moins une image par ensemble.")
        validation_ids = set(random.Random(CFG["seed"]).sample(sorted(expected_train), validation_count))
        for path in sorted(train_paths, key=lambda item: item.stem):
            split = "val" if path.stem in validation_ids else "train"
            candidates[split].append((path, f"div2k_{path.stem}", "DIV2K_train_HR"))
        for path in sorted(valid_paths, key=lambda item: item.stem):
            candidates["test"].append((path, f"div2k_{path.stem}", "DIV2K_valid_HR"))
    else:
        for split in candidates:
            folder = root / split
            paths = sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
            for path in paths:
                relative = path.relative_to(folder)
                if len(relative.parts) < 2:
                    raise ValueError(f"Dossier de scène manquant : {path}")
                candidates[split].append((path, relative.parts[0], split))
    return candidates

def audit_dataset(root):
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(
            f"Dataset introuvable : {root}. Vérifier Add Input ou le chemin fourni par KaggleHub."
        )
    records, errors, seen_pixels, seen_scenes = [], [], {}, {}
    candidates = dataset_candidates(root)
    for split in ("train", "val", "test"):
        if not candidates[split]:
            errors.append(f"Aucune image JPEG/PNG pour l'ensemble {split}")
        for path, scene, source_split in tqdm(candidates[split], desc=f"Audit {split}"):
            if scene in seen_scenes and seen_scenes[scene] != split:
                errors.append(f"Scène {scene!r} présente dans plusieurs ensembles.")
            seen_scenes[scene] = split
            try:
                rgb = load_rgb(path)
                height, width, _ = rgb.shape
                if min(height, width) < CFG["eval_size"]:
                    raise ValueError(f"Trop petite : {width}x{height}, minimum {CFG['eval_size']} par côté.")
                pixel_hash = hashlib.sha256(str(rgb.shape).encode() + rgb.tobytes()).hexdigest()
                if pixel_hash in seen_pixels:
                    errors.append(f"Doublon exact : {path} et {seen_pixels[pixel_hash]}")
                seen_pixels[pixel_hash] = str(path)
                records.append({
                    "path": path.relative_to(root).as_posix(), "split": split,
                    "scene": scene, "source_split": source_split, "width": width, "height": height,
                    "file_sha256": file_sha256(path), "pixel_sha256": pixel_hash,
                })
            except Exception as exc:
                errors.append(f"{path}: {exc}")
    if errors:
        raise ValueError("Corriger le dataset avant de continuer :\n" + "\n".join(errors[:30]))
    return root, pd.DataFrame(records)

DATA_ROOT, manifest = audit_dataset(DATA_ROOT)
manifest_text = manifest.to_csv(index=False, lineterminator="\n")
MANIFEST_HASH = hashlib.sha256(manifest_text.encode("utf-8")).hexdigest()
print("Empreinte du manifeste :", MANIFEST_HASH)
display(manifest.groupby("split").agg(images=("path", "count"), groups=("scene", "nunique")))
if CFG["dataset_layout"] == "div2k":
    print("DIV2K : groupes par identifiant d'image, scènes non annotées ; validation publique réservée au test interne.")
print("Vérifier manuellement les quasi-doublons et les informations de source/licence.")


## 5. Dataset synthétique et aperçu de validation

Entraînement : recadrages aléatoires et rotations/miroirs de l'image propre, puis
ajout de bruit gaussien de niveau aléatoire entre 0 et 50 sur l'échelle 0–255.
Validation/test : recadrage central fixe et bruits déterministes à 15, 25 et 50.
Les valeurs RGB flottantes sont dans `[0, 1]` et l'entrée bruitée est écrêtée.
L'évaluation porte donc sur des **recadrages synthétiques**, pas sur toute la photo.
La validation moyenne donne le même poids à chaque image et niveau de bruit.
Les images restent en uint8 jusqu’au recadrage, puis sont converties en
flottants. `RGB_CACHE_MIB` borne un cache LRU par dataset/processus pour
limiter les redécodages, particulièrement sur les trois bruits de validation.


In [ ]:
class SyntheticDenoisingDataset(Dataset):
    def __init__(self, frame, training=False):
        self.frame = frame.reset_index(drop=True)
        self.training = training
        self.epoch = 0
        self._rgb_cache = OrderedDict()
        self._cache_bytes = 0
        self._cache_limit = max(0, int(RGB_CACHE_MIB * 2**20))

    def _read_rgb(self, relative_path):
        if relative_path in self._rgb_cache:
            self._rgb_cache.move_to_end(relative_path)
            return self._rgb_cache[relative_path]
        rgb = load_rgb(DATA_ROOT / relative_path)
        if rgb.nbytes <= self._cache_limit:
            while self._rgb_cache and self._cache_bytes + rgb.nbytes > self._cache_limit:
                _, removed = self._rgb_cache.popitem(last=False)
                self._cache_bytes -= removed.nbytes
            self._rgb_cache[relative_path] = rgb
            self._cache_bytes += rgb.nbytes
        return rgb

    def __len__(self):
        repeat = CFG["patches_per_image"] if self.training else len(CFG["eval_sigmas"])
        return len(self.frame) * repeat

    def __getitem__(self, index):
        if self.training:
            image_index = index // CFG["patches_per_image"]
            rng = np.random.default_rng(np.random.SeedSequence([CFG["seed"], self.epoch, index, 1]))
        else:
            image_index = index // len(CFG["eval_sigmas"])
            rng = np.random.default_rng(np.random.SeedSequence([CFG["seed"], index, 2]))
        row = self.frame.iloc[image_index]
        clean = self._read_rgb(row["path"])  # Garder l’image 2K en uint8 jusqu’au recadrage.
        size = CFG["patch_size"] if self.training else CFG["eval_size"]
        h, w = clean.shape[:2]
        if self.training:
            top, left = int(rng.integers(h - size + 1)), int(rng.integers(w - size + 1))
        else:
            top, left = (h - size) // 2, (w - size) // 2
        clean = clean[top:top + size, left:left + size]
        if self.training:
            clean = np.rot90(clean, int(rng.integers(4)))
            if rng.random() < 0.5:
                clean = np.fliplr(clean)
            sigma = float(rng.uniform(CFG["train_sigma_min"], CFG["train_sigma_max"]))
        else:
            sigma = CFG["eval_sigmas"][index % len(CFG["eval_sigmas"])]
        clean = clean.astype(np.float32) / 255.0
        noise = rng.normal(0.0, sigma / 255.0, clean.shape).astype(np.float32)
        noisy = np.clip(clean + noise, 0.0, 1.0)
        to_tensor = lambda array: torch.from_numpy(np.ascontiguousarray(array.transpose(2, 0, 1)))
        return {"noisy": to_tensor(noisy), "clean": to_tensor(clean),
                "path": row["path"], "scene": row["scene"], "sigma": float(sigma)}

train_dataset = SyntheticDenoisingDataset(manifest[manifest["split"] == "train"], training=True)
val_dataset = SyntheticDenoisingDataset(manifest[manifest["split"] == "val"])
test_dataset = SyntheticDenoisingDataset(manifest[manifest["split"] == "test"])
loader_generator = torch.Generator().manual_seed(CFG["seed"])
train_loader = DataLoader(
    train_dataset, batch_size=CFG["batch_size"], shuffle=True,
    num_workers=CFG["num_workers"], pin_memory=DEVICE.type == "cuda",
    generator=loader_generator,
    persistent_workers=False,  # Recréer les workers à chaque époque pour propager epoch.
)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

def as_hwc(tensor):
    return tensor.detach().cpu().float().permute(1, 2, 0).numpy()

preview = val_dataset[1 if len(CFG["eval_sigmas"]) > 1 else 0]
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(as_hwc(preview["clean"]))
axes[0].set_title("Référence propre — validation")
axes[1].imshow(as_hwc(preview["noisy"]))
axes[1].set_title(f"Entrée synthétique — sigma {preview['sigma']:g}/255")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 6. Modèle AyyPhotoDenoiser

Petit CNN résiduel sans changement de résolution : huit convolutions 3×3,
48 canaux internes et activations ReLU par défaut. Le réseau prédit une
correction à soustraire à l'entrée. La dernière couche est initialisée à zéro :
avant apprentissage, la sortie correspond à l'image d'entrée.

La perte L1 porte sur la sortie brute, sans écrêtage pendant l'entraînement.
Les sorties destinées aux métriques et aux images sont limitées à `[0, 1]`.
Ce choix reste une première architecture à comparer aux baselines.


In [ ]:
class AyyPhotoDenoiser(nn.Module):
    def __init__(self, channels=48, depth=8):
        super().__init__()
        layers = [nn.Conv2d(3, channels, 3, padding=1), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True)]
        layers += [nn.Conv2d(channels, 3, 3, padding=1)]
        self.network = nn.Sequential(*layers)
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, image):
        return image - self.network(image)

model = AyyPhotoDenoiser(CFG["channels"], CFG["depth"]).to(DEVICE)
PARAMETERS = sum(parameter.numel() for parameter in model.parameters())
print(f"Paramètres entraînables : {PARAMETERS:,}")
print(f"Poids FP32 seuls, estimation : {PARAMETERS * 4 / 2**20:.2f} Mio")
print("La RAM d'inférence et d'entraînement sera supérieure à la taille des poids.")
with torch.inference_mode():
    probe = model(torch.zeros(1, 3, 17, 19, device=DEVICE))
    assert probe.shape == (1, 3, 17, 19) and torch.isfinite(probe).all()
del probe
print("Vérification du forward et des dimensions impaires réussie avant entraînement.")


## 7. Mesures et baseline classique

Comparer les trois méthodes sur **les mêmes entrées bruitées et références** :
image bruitée inchangée, OpenCV Non-Local Means couleur et CNN.
PSNR et SSIM sont calculés sur RGB `[0, 1]` ; SSIM utilise `channel_axis=-1`.
Le paramétrage OpenCV ci-dessous est une baseline fixe à documenter, pas un optimum.

Les latences mesurent le traitement d'un recadrage : transfert de l'entrée GPU
effectué avant chronométrage, synchronisation CUDA autour de l'inférence.
Le décodage, les métriques, le réseau et l'enregistrement sont exclus.
La latence OpenCV inclut ses conversions uint8 et RGB/BGR.


In [ ]:
def image_metrics(clean, restored):
    return {
        "psnr": float(peak_signal_noise_ratio(clean, restored, data_range=1.0)),
        "ssim": float(structural_similarity(clean, restored, data_range=1.0, channel_axis=-1)),
    }

def classic_denoise(noisy, sigma):
    rgb8 = np.rint(np.clip(noisy, 0, 1) * 255).astype(np.uint8)
    bgr = cv2.cvtColor(rgb8, cv2.COLOR_RGB2BGR)
    strength = max(1.0, float(sigma) * 0.8)
    result = cv2.fastNlMeansDenoisingColored(
        bgr, None, h=strength, hColor=strength, templateWindowSize=7, searchWindowSize=21,
    )
    return cv2.cvtColor(result, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

def noisy_validation_baseline(dataset):
    records = []
    for index in range(len(dataset)):
        sample = dataset[index]
        records.append(image_metrics(as_hwc(sample["clean"]), as_hwc(sample["noisy"])))
    if not records:
        raise ValueError("Validation vide.")
    return {key: float(np.mean([row[key] for row in records])) for key in ("psnr", "ssim")}

def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

@torch.inference_mode()
def evaluate_cnn(loader):
    model.eval()
    records = []
    for batch in loader:
        noisy = batch["noisy"].to(DEVICE)
        raw_prediction = model(noisy).float().cpu()
        prediction = raw_prediction.clamp(0, 1)
        for index in range(prediction.shape[0]):
            scores = image_metrics(as_hwc(batch["clean"][index]), as_hwc(prediction[index]))
            scores["l1"] = float(nn.functional.l1_loss(raw_prediction[index], batch["clean"][index]))
            scores["l1_clipped"] = float(nn.functional.l1_loss(prediction[index], batch["clean"][index]))
            records.append(scores)
    if not records:
        raise ValueError("Aucun échantillon de validation.")
    return {name: float(np.mean([row[name] for row in records])) for name in records[0]}

@torch.inference_mode()
def benchmark(dataset, description):
    model.eval()
    warmup = dataset[0]["noisy"].unsqueeze(0).to(DEVICE)
    for _ in range(3):
        model(warmup)
    synchronize()
    records = []
    for index in tqdm(range(len(dataset)), desc=description):
        sample = dataset[index]
        clean, noisy = as_hwc(sample["clean"]), as_hwc(sample["noisy"])
        common = {"path": sample["path"], "scene": sample["scene"], "sigma": sample["sigma"]}
        records.append({**common, "method": "noisy", **image_metrics(clean, noisy), "latency_ms": 0.0})
        started = time.perf_counter()
        classic = classic_denoise(noisy, sample["sigma"])
        elapsed = (time.perf_counter() - started) * 1000
        records.append({**common, "method": "opencv", **image_metrics(clean, classic), "latency_ms": elapsed})
        tensor = sample["noisy"].unsqueeze(0).to(DEVICE)
        synchronize()
        started = time.perf_counter()
        prediction = model(tensor).clamp(0, 1)
        synchronize()
        elapsed = (time.perf_counter() - started) * 1000
        restored = as_hwc(prediction[0])
        records.append({**common, "method": "cnn", **image_metrics(clean, restored), "latency_ms": elapsed})
    return pd.DataFrame(records)

def aggregate(frame):
    return frame.groupby(["method", "sigma"]).agg(
        psnr=("psnr", "mean"), ssim=("ssim", "mean"),
        latency_p50_ms=("latency_ms", "median"),
        latency_p95_ms=("latency_ms", lambda series: series.quantile(0.95)),
        samples=("path", "count"),
    ).reset_index()


## 8. Entraînement, checkpoints et reprise

**L’entraînement est activé (`RUN_TRAINING = True`) : exécuter cette cellule le lance.**
Pour préparer les données sans entraîner, remettre cette option à `False`.
La meilleure époque est choisie uniquement à partir du PSNR moyen de validation.
AdamW optimise la perte L1 ; le taux d'apprentissage est réduit si la validation
stagne et un arrêt anticipé limite les époques inutiles.

Chaque époque terminée sauvegarde `last.ckpt` avec les poids, l'optimiseur,
le scheduler, le scaler AMP, l'historique et les états aléatoires. Une interruption
au milieu d'une époque oblige à reprendre cette époque depuis son début.
`best_weights.pth` conserve les meilleurs poids de validation. Aucun fichier
`1.0.0` ni déploiement n'est créé automatiquement.

Pour réouvrir les rapports ou effectuer le test final sans réentraîner : mettre
`RUN_TRAINING = False` et renseigner `RESUME_RUN_DIR`. La configuration scientifique et le
manifeste doivent correspondre ; epochs et num_workers peuvent varier. Après un test final, démarrer une nouvelle
expérience pour d'autres réglages ; ne pas réentraîner dans le même dossier.


In [ ]:
# Seuls les artefacts du modèle sont copiés/exportés, jamais les photos du dataset.
RUN_ARTIFACTS = (
    "config.json", "manifest.csv", "data_audit.json", "environment.txt", "history.csv",
    "best_weights.pth", "last.ckpt", "interrupted_weights.pth", "model_bundle.pt",
    "summary.json", "status.json", "baseline_validation.json", "validation_metrics.csv", "test_metrics.csv",
    "test_protocol.json",
)

def writable_run_copy(directory, output_root):
    directory, output_root = Path(directory).resolve(), Path(output_root).resolve()
    if not (directory / "config.json").is_file() or not (directory / "best_weights.pth").is_file():
        raise FileNotFoundError("L’expérience doit contenir config.json et best_weights.pth.")
    if directory == output_root or output_root in directory.parents:
        return directory
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    destination = output_root / f"{directory.name}_restored_{stamp}"
    destination.mkdir(parents=True, exist_ok=False)
    for name in RUN_ARTIFACTS:
        artifact = directory / name
        if artifact.is_file():
            shutil.copyfile(artifact, destination / name)
    return destination

def export_run_archive(directory):
    directory = Path(directory)
    if (directory / "best_weights.pth").is_file():
        # Un fichier unique suffit ensuite pour tester une photo, sans DIV2K.
        bundle = {
            "format_version": 1, "architecture": "residual_rgb_cnn",
            "model_state_dict": torch.load(directory / "best_weights.pth", map_location="cpu", weights_only=True),
            "config": {"channels": CFG["channels"], "depth": CFG["depth"]},
            "preprocessing": {"color_space": "sRGB", "input_range": [0, 1], "output_clamp": [0, 1]},
            "model_version": CFG["model_version"],
            "source_weights_sha256": file_sha256(directory / "best_weights.pth"),
        }
        atomic_torch_save(bundle, directory / "model_bundle.pt")
    archive = directory.parent / f"{directory.name}_export.zip"
    temporary = archive.with_name(archive.name + ".tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as output:
        for name in RUN_ARTIFACTS:
            file = directory / name
            if file.is_file():
                output.write(file, arcname=f"{directory.name}/{name}")
    temporary.replace(archive)
    return archive


In [ ]:
def write_json(path, payload):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    temporary.replace(path)

def atomic_torch_save(payload, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)

def rng_state():
    numpy_state = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": [numpy_state[0], numpy_state[1].tolist(), numpy_state[2], numpy_state[3], numpy_state[4]],
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else [],
        "loader": loader_generator.get_state(),
    }

def restore_rng(state):
    random.setstate(state["python"])
    name, keys, position, has_gauss, cached_gauss = state["numpy"]
    np.random.set_state((name, np.array(keys, dtype=np.uint32), position, has_gauss, cached_gauss))
    torch.set_rng_state(state["torch"])
    loader_generator.set_state(state["loader"])
    if DEVICE.type == "cuda" and state["cuda"]:
        available = torch.cuda.device_count()
        for index, cuda_state in enumerate(state["cuda"][:available]):
            torch.cuda.set_rng_state(cuda_state, device=index)
        if len(state["cuda"]) != available:
            print("Nombre de GPU différent de la sauvegarde : états compatibles restaurés, reproductibilité inter-matériel non garantie.")

def verify_existing_run(directory):
    saved = json.loads((directory / "config.json").read_text(encoding="utf-8"))
    if saved["manifest_sha256"] != MANIFEST_HASH:
        raise ValueError("Le dataset a changé : créer une nouvelle expérience.")
    differences = [key for key in CFG if key not in {"epochs", "num_workers"}
                   and CFG[key] != saved["config"].get(key)]
    if differences:
        raise ValueError(f"Configuration différente : {differences}. Reprendre celle de config.json.")
    return saved

RUN_DIR = None
history = []
HAS_TRAINED_MODEL = False
baseline_validation = None
if RESUME_RUN_DIR:
    selected_run = Path(RESUME_RUN_DIR).expanduser().resolve()
    verify_existing_run(selected_run)
    RUN_DIR = writable_run_copy(selected_run, OUTPUT_ROOT)
    if RUN_DIR != selected_run:
        print("Artefacts de l’expérience restaurés dans un dossier accessible en écriture :", RUN_DIR)

if RUN_TRAINING:
    if RUN_DIR is None:
        seed_everything(CFG["seed"])
        loader_generator.manual_seed(CFG["seed"])
        model = AyyPhotoDenoiser(CFG["channels"], CFG["depth"]).to(DEVICE)
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        RUN_DIR = OUTPUT_ROOT.expanduser().resolve() / f"denoise_{stamp}"
        RUN_DIR.mkdir(parents=True, exist_ok=False)
    if (RUN_DIR / "test_metrics.csv").exists():
        raise ValueError("Cette expérience a déjà été testée. Créer une nouvelle expérience pour réentraîner.")

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["learning_rate"], weight_decay=CFG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    start_epoch, best_psnr, bad_epochs = 0, -math.inf, 0
    if RESUME_RUN_DIR:
        checkpoint = torch.load(RUN_DIR / "last.ckpt", map_location="cpu", weights_only=True)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        if AMP_ENABLED and checkpoint["scaler_state_dict"]:
            scaler.load_state_dict(checkpoint["scaler_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        best_psnr, bad_epochs = checkpoint["best_val_psnr"], checkpoint["bad_epochs"]
        history = checkpoint["history"]
        restore_rng(checkpoint["rng_state"])
        if not (RUN_DIR / "best_weights.pth").is_file():
            raise FileNotFoundError("best_weights.pth manquant : restaurer le dossier complet de l'expérience.")
    if start_epoch >= CFG["epochs"]:
        print("Les époques demandées sont déjà terminées : chargement du meilleur modèle sans réentraîner.")
    stop_epoch = start_epoch if bad_epochs >= CFG["patience"] else CFG["epochs"]
    if bad_epochs >= CFG["patience"]:
        print("L’expérience avait déjà atteint l’arrêt anticipé ; créer une nouvelle expérience pour ajuster les paramètres.")

    baseline_validation = noisy_validation_baseline(val_dataset)
    write_json(RUN_DIR / "baseline_validation.json", baseline_validation)
    print(f"Référence avant apprentissage — entrée bruitée : {baseline_validation['psnr']:.3f} dB")
    write_json(RUN_DIR / "config.json", {
        "config": CFG, "manifest_sha256": MANIFEST_HASH, "data_root": str(DATA_ROOT),
        "environment": ENVIRONMENT, "architecture": "residual_rgb_cnn",
        "dataset_source": {key: value for key, value in DATASET_PROVENANCE.items() if key != "inventory"}
                          if DATASET_PROVENANCE is not None else None,
        "input_range": [0, 1], "color_space": "sRGB", "noise": "clipped_additive_gaussian",
    })
    (RUN_DIR / "manifest.csv").write_text(manifest_text, encoding="utf-8")
    write_json(RUN_DIR / "data_audit.json", {
        "manifest_sha256": MANIFEST_HASH,
        "counts": {split: int((manifest["split"] == split).sum()) for split in ("train", "val", "test")},
        "exact_duplicates": 0, "group_overlap": 0,
        "scene_overlap": "not_annotated" if CFG["dataset_layout"] == "div2k" else 0,
        "grouping": "image_id_only" if CFG["dataset_layout"] == "div2k" else "scene_folder",
        "near_duplicates": "manual_review_required", "licenses": "user_verification_required",
    })
    frozen = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
    (RUN_DIR / "environment.txt").write_text(frozen, encoding="utf-8")
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    print("Dossier de l'expérience :", RUN_DIR)
    try:
        for epoch in range(start_epoch, stop_epoch):
            epoch_started = time.perf_counter()
            train_dataset.epoch = epoch
            model.train()
            total_loss, samples = 0.0, 0
            learning_rate = float(optimizer.param_groups[0]["lr"])
            progress = tqdm(train_loader, desc=f"Époque {epoch + 1}/{CFG['epochs']}")
            for batch in progress:
                noisy = batch["noisy"].to(DEVICE, non_blocking=True)
                clean = batch["clean"].to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                    prediction = model(noisy)
                    loss = nn.functional.l1_loss(prediction, clean)
                if not torch.isfinite(loss):
                    raise FloatingPointError("Perte non finie : arrêter et vérifier les données/configuration.")
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                count = noisy.shape[0]
                total_loss += float(loss.detach()) * count
                samples += count
                progress.set_postfix(l1=total_loss / samples)
            validation = evaluate_cnn(val_loader)
            if not all(math.isfinite(value) for value in validation.values()):
                raise FloatingPointError("Métrique de validation non finie.")
            scheduler.step(validation["psnr"])
            improved = validation["psnr"] > best_psnr
            if improved:
                best_psnr, bad_epochs = validation["psnr"], 0
                atomic_torch_save(model.state_dict(), RUN_DIR / "best_weights.pth")
            else:
                bad_epochs += 1
            row = {
                "epoch": epoch + 1, "train_l1": total_loss / samples,
                "val_l1": validation["l1"], "val_l1_clipped": validation["l1_clipped"],
                "val_psnr": validation["psnr"],
                "val_gain_vs_noisy_db": validation["psnr"] - baseline_validation["psnr"],
                "val_ssim": validation["ssim"], "learning_rate": learning_rate,
                "seconds": time.perf_counter() - epoch_started,
                "gpu_peak_allocated_mib": torch.cuda.max_memory_allocated() / 2**20
                                          if DEVICE.type == "cuda" else None,
            }
            history.append(row)
            atomic_torch_save({
                "epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "best_val_psnr": best_psnr, "bad_epochs": bad_epochs,
                "history": history, "rng_state": rng_state(), "config": CFG,
                "manifest_sha256": MANIFEST_HASH,
            }, RUN_DIR / "last.ckpt")
            pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)
            write_json(RUN_DIR / "status.json", {
                "state": "training", "last_completed_epoch": epoch + 1,
                "best_validation_psnr_db": best_psnr,
                "updated_at_utc": datetime.now(timezone.utc).isoformat(),
            })
            if EXPORT_EACH_EPOCH:
                export_run_archive(RUN_DIR)
            print(f"Époque {epoch + 1}: L1 train={row['train_l1']:.5f}, PSNR val={row['val_psnr']:.3f} dB, SSIM={row['val_ssim']:.4f}")
            if bad_epochs >= CFG["patience"]:
                print("Arrêt anticipé : validation sans amélioration.")
                break
    except KeyboardInterrupt:
        atomic_torch_save(model.state_dict(), RUN_DIR / "interrupted_weights.pth")
        write_json(RUN_DIR / "status.json", {
            "state": "interrupted", "last_completed_epoch": len(history),
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
            "interrupted_weights_usage": "inference_or_finetuning_only; resume uses last.ckpt",
        })
        archive = export_run_archive(RUN_DIR)
        print("Interruption : artefacts exportés dans", archive)
        print("Pour une reprise, utiliser last.ckpt ; les poids interrompus ne constituent pas un état de reprise exacte.")
        raise
    HAS_TRAINED_MODEL = True
elif RUN_DIR is not None:
    history = pd.read_csv(RUN_DIR / "history.csv").to_dict(orient="records")
    HAS_TRAINED_MODEL = True
    print("Chargement d'une expérience existante, sans entraînement :", RUN_DIR)
else:
    print("Entraînement désactivé. Mettre RUN_TRAINING = True dans la configuration pour le lancer.")

if HAS_TRAINED_MODEL:
    model.load_state_dict(torch.load(RUN_DIR / "best_weights.pth", map_location=DEVICE, weights_only=True))
    model.eval()
    print("Meilleurs poids de validation chargés.")
    if baseline_validation is None:
        baseline_validation = noisy_validation_baseline(val_dataset)
        write_json(RUN_DIR / "baseline_validation.json", baseline_validation)
    write_json(RUN_DIR / "status.json", {
        "state": "ready", "last_completed_epoch": len(history),
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
    })
    archive = export_run_archive(RUN_DIR)
    print("Archive à sauvegarder via les sorties Kaggle ou à télécharger :", archive)


## 9. Courbes et comparaison de validation

Les nouvelles expériences utilisent une L1 de validation **brute**, comme
au train ; `val_l1_clipped` conserve aussi la valeur après écrêtage. Les
anciennes courbes `val_l1` utilisaient la sortie écrêtée : ne pas comparer
ces pertes directement entre anciennes et nouvelles expériences. Les PSNR
et SSIM gardent leur protocole sur la sortie écrêtée.

Observer les pertes et les métriques, puis inspecter les détails fins, les
couleurs, les visages et les textes. Une hausse du PSNR ne suffit pas à prouver
un meilleur rendu dans tous les cas. Les figures sont enregistrées sans
métadonnées EXIF dans le dossier de l'expérience.


In [ ]:
if HAS_TRAINED_MODEL:
    history_frame = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history_frame["epoch"], history_frame["train_l1"], label="train")
    axes[0].plot(history_frame["epoch"], history_frame["val_l1"], label="validation")
    axes[0].set_title("Perte L1")
    axes[0].legend()
    axes[1].plot(history_frame["epoch"], history_frame["val_psnr"])
    axes[1].set_title("PSNR validation (dB)")
    axes[2].plot(history_frame["epoch"], history_frame["val_ssim"])
    axes[2].set_title("SSIM validation")
    for ax in axes:
        ax.set_xlabel("Époque")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(RUN_DIR / "learning_curves.png", dpi=140)
    plt.show()

    validation_metrics = benchmark(val_dataset, "Benchmark validation")
    validation_metrics.to_csv(RUN_DIR / "validation_metrics.csv", index=False)
    display(aggregate(validation_metrics))
    delta = validation_metrics.pivot(index=["path", "sigma"], columns="method", values="psnr")
    print(f"Gain PSNR moyen CNN / entrée bruitée : {(delta['cnn'] - delta['noisy']).mean():+.3f} dB")
    print(f"Gain PSNR moyen CNN / OpenCV : {(delta['cnn'] - delta['opencv']).mean():+.3f} dB")
else:
    print("Courbes et benchmark disponibles après l'entraînement ou le chargement d'une expérience.")


In [ ]:
@torch.inference_mode()
def comparison_figure(dataset, output_path, image_count=3):
    model.eval()
    noise_index = min(1, len(CFG["eval_sigmas"]) - 1)
    count = min(image_count, len(dataset.frame))
    fig, axes = plt.subplots(count, 4, figsize=(14, 3.5 * count), squeeze=False)
    for row_index in range(count):
        sample = dataset[row_index * len(CFG["eval_sigmas"]) + noise_index]
        clean, noisy = as_hwc(sample["clean"]), as_hwc(sample["noisy"])
        classical = classic_denoise(noisy, sample["sigma"])
        restored = as_hwc(model(sample["noisy"].unsqueeze(0).to(DEVICE))[0].clamp(0, 1))
        for ax, image, title in zip(axes[row_index], (clean, noisy, classical, restored),
                                    ("Référence", "Entrée bruitée", "OpenCV", "CNN AyyPhoto")):
            ax.imshow(image)
            if title != "Référence":
                scores = image_metrics(clean, image)
                title += f"\n{scores['psnr']:.2f} dB | SSIM {scores['ssim']:.3f}"
            ax.set_title(title)
            ax.axis("off")
    fig.suptitle(f"Validation synthétique — sigma {CFG['eval_sigmas'][noise_index]:g}/255")
    fig.tight_layout()
    fig.savefig(output_path, dpi=140)
    plt.show()

if HAS_TRAINED_MODEL:
    comparison_figure(val_dataset, RUN_DIR / "comparison_validation.png")


## 10. Test final — optionnel, après choix définitif du modèle

Laisser `RUN_FINAL_TEST = False` pendant les essais. Une fois l'architecture,
les hyperparamètres et les poids choisis avec la validation, activer cette option
et exécuter cette cellule. Un rapport existant est relu sans recalculer le test.
Les résultats du test ne doivent pas guider les réglages suivants ; utiliser un
nouveau jeu de confirmation indépendant si le développement reprend.


In [ ]:
test_metrics = None
if HAS_TRAINED_MODEL:
    expected_protocol = {
        "weights_sha256": file_sha256(RUN_DIR / "best_weights.pth"),
        "manifest_sha256": MANIFEST_HASH,
        "eval_size": CFG["eval_size"], "eval_sigmas": CFG["eval_sigmas"],
    }
    metrics_path, protocol_path = RUN_DIR / "test_metrics.csv", RUN_DIR / "test_protocol.json"
    if metrics_path.exists():
        if not protocol_path.exists():
            print("Ancien rapport de test sans empreinte : conservé, mais non attribué automatiquement aux poids actuels.")
        elif json.loads(protocol_path.read_text(encoding="utf-8")) != expected_protocol:
            raise ValueError("Le rapport de test existant ne correspond pas aux poids ou au protocole courant.")
        else:
            test_metrics = pd.read_csv(metrics_path)
            print("Rapport final existant vérifié et chargé.")
            display(aggregate(test_metrics))
    elif RUN_FINAL_TEST:
        test_metrics = benchmark(test_dataset, "Test final gelé")
        test_metrics.to_csv(metrics_path, index=False)
        write_json(protocol_path, expected_protocol)
        display(aggregate(test_metrics))
    else:
        print("Test final désactivé : la validation suffit pendant les essais.")
elif RUN_FINAL_TEST:
    raise RuntimeError("Entraîner ou charger une expérience avant le test final.")
else:
    print("Aucun test final demandé.")


## 11. Photo réelle optionnelle — inférence par tuiles

Renseigner `REAL_PHOTO_PATH` pour essayer une photo personnelle autorisée.
Le traitement conserve la taille de l'image après orientation EXIF et écrit un
nouveau PNG sRGB sans recopier ses métadonnées. La source n'est pas remplacée.
Les tuiles utilisent une marge au moins égale au rayon du champ réceptif du CNN
pour éviter des coutures artificielles. La photo complète reste en RAM CPU ;
seules les tuiles passent sur le GPU.

Sans référence propre alignée, aucun PSNR/SSIM n'est calculé. Inspecter le
résultat visuellement : cette expérience synthétique peut mal se comporter sur
le bruit réel. Elle ne vise ni la super-résolution ni la correction d'exposition.


In [ ]:
@torch.inference_mode()
def infer_tiled(rgb):
    model.eval()
    tensor = torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1))).float() / 255.0
    _, height, width = tensor.shape
    result = torch.empty_like(tensor)
    tile, halo = CFG["tile_size"], CFG["tile_halo"]
    for top in tqdm(range(0, height, tile), desc="Inférence par tuiles"):
        bottom = min(top + tile, height)
        for left in range(0, width, tile):
            right = min(left + tile, width)
            y0, y1 = max(0, top - halo), min(height, bottom + halo)
            x0, x1 = max(0, left - halo), min(width, right + halo)
            patch = tensor[:, y0:y1, x0:x1].unsqueeze(0).to(DEVICE)
            restored = model(patch)[0].clamp(0, 1).cpu()
            result[:, top:bottom, left:right] = restored[:, top - y0:bottom - y0, left - x0:right - x0]
    return np.rint(as_hwc(result) * 255).astype(np.uint8)

if REAL_PHOTO_PATH:
    if not HAS_TRAINED_MODEL:
        raise RuntimeError("Entraîner ou charger une expérience avant l'inférence réelle.")
    source_path = Path(REAL_PHOTO_PATH).expanduser().resolve()
    output_path = RUN_DIR / "real_photo_denoised.png"
    if source_path == output_path.resolve():
        raise ValueError("Choisir une image source différente du fichier de sortie.")
    original = load_rgb(source_path)
    enhanced = infer_tiled(original)
    srgb_profile = ImageCms.ImageCmsProfile(ImageCms.createProfile("sRGB")).tobytes()
    Image.fromarray(enhanced).save(output_path, icc_profile=srgb_profile)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, photo, title in zip(axes, (original, enhanced), ("Original réel", "Résultat CNN")):
        ax.imshow(photo)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    print("Résultat enregistré :", output_path)
else:
    print("Inférence réelle optionnelle : renseigner REAL_PHOTO_PATH pour l'utiliser.")


## 12. Rapport à me transmettre

Après exécution, transmettre **`summary.json`**, **`learning_curves.png`** et
**`comparison_validation.png`**. Les CSV détaillent les résultats par image et
niveau de bruit. Les comparaisons visuelles contiennent des extraits de tes
images : choisir ce que tu souhaites partager. Aucun envoi automatique n'a lieu.

`last.ckpt` et `best_weights.pth` servent à conserver/reprendre le modèle ; ils
ne sont pas nécessaires pour une première analyse des résultats. Sur Kaggle,
enregistrer une version avec ses sorties ou télécharger l’archive `*_export.zip`.
L’archive contient les artefacts et métriques, sans les photos du dataset
ni les comparaisons PNG. `model_bundle.pt` suffit au notebook d’inférence
02 ; `last.ckpt` sert à reprendre l’entraînement.

Le rapport distingue validation et test final. Un gain synthétique ne valide
pas encore la qualité sur de vrais smartphones, la performance Android ni le
temps total d'un parcours cloud. Aucune promotion en production n'est automatique.


In [ ]:
if HAS_TRAINED_MODEL:
    best_row = max(history, key=lambda row: row["val_psnr"])
    report = {
        "status": "research_candidate",
        "model_version": CFG["model_version"],
        "run_directory": str(RUN_DIR),
        "config": CFG,
        "environment": ENVIRONMENT,
        "parameters": PARAMETERS,
        "best_weights_sha256": file_sha256(RUN_DIR / "best_weights.pth"),
        "manifest_sha256": MANIFEST_HASH,
        "dataset_source": {key: value for key, value in DATASET_PROVENANCE.items() if key != "inventory"}
                          if DATASET_PROVENANCE is not None else None,
        "image_counts": {split: int((manifest["split"] == split).sum()) for split in ("train", "val", "test")},
        "epochs_completed": len(history),
        "training_seconds_recorded": float(sum(row["seconds"] for row in history)),
        "best_epoch": int(best_row["epoch"]),
        "best_validation_psnr_db": float(best_row["val_psnr"]),
        "validation_noisy_baseline": baseline_validation,
        "best_validation_gain_vs_noisy_db": float(best_row["val_psnr"]) - baseline_validation["psnr"],
        "validation": aggregate(validation_metrics).to_dict(orient="records"),
        "test_performed": test_metrics is not None,
        "test": aggregate(test_metrics).to_dict(orient="records") if test_metrics is not None else None,
        "protocol": {
            "evaluation": "fixed_center_crops_with_deterministic_clipped_gaussian_noise",
            "color_space": "sRGB", "pixel_range": [0, 1],
            "selection": "highest_mean_validation_psnr",
            "latency": "patch_processing_only_no_network_or_decoding",
            "opencv_strength": "h=hColor=max(1,sigma*0.8)",
            "dataset_layout": CFG["dataset_layout"],
            "test_source": "DIV2K public validation; not official DIV2K test"
                           if CFG["dataset_layout"] == "div2k" else "local test split",
        },
        "limitations": [
            "Synthetic Gaussian noise; smartphone noise not yet validated.",
            "Evaluation on center crops, not full-resolution photos.",
            "Near-duplicates and image rights require manual review.",
            "DIV2K grouping uses image identifiers, not verified scene labels; academic-research terms apply when using DIV2K.",
            "No Android runtime or cloud latency benchmark performed.",
            "No automatic production approval.",
        ],
    }
    write_json(RUN_DIR / "summary.json", report)
    print(json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False))
    archive = export_run_archive(RUN_DIR)
    print("Archive actualisée :", archive)
    print("À transmettre : summary.json, learning_curves.png, comparison_validation.png")
else:
    print("Aucun rapport d'entraînement : aucun modèle n'a été entraîné ou chargé.")


## Références techniques


- [Cahier des charges AyyPhoto](../../README_AyyPhoto.md) — disponible dans le dépôt.
- [DIV2K : données, structure, conditions et citations](https://data.vision.ee.ethz.ch/cvl/DIV2K/)
- [Notebooks Jupyter dans VS Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks)
- [Notebooks Kaggle](https://www.kaggle.com/docs/notebooks)
- [PyTorch : sauvegarde et chargement](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)
- [PyTorch : précision mixte](https://docs.pytorch.org/docs/stable/notes/amp_examples.html)
- [scikit-image : PSNR et SSIM](https://scikit-image.org/docs/stable/api/skimage.metrics.html)
